# Foundation-nnU-Net — Colab Training (Legacy Notebook)

> Legacy / non-authoritative notebook.
>
> This notebook is preserved as historical operational context only. It is **not**
> the current source of truth for training workflow, output locations, hybrid status,
> or paper-ready evaluation.
>
> Before using anything below, defer to:
> - `RECOVERY_TODO.md`
> - `AGENT_CONTEXT.md`
> - `DECISIONS.md`
> - `VALIDATION_CHECKLIST.md`
> - current code and tests
>
> Known stale content remains below, including legacy `results/` handling and
> hybrid-first workflow cells.
>
> Current authoritative conventions are:
> - trusted dataset root: `data/processed/pneumothorax_trusted_v1`
> - authoritative outputs: `artifacts/runs/`
> - notebook outputs are non-authoritative unless fully traceable
> - Foundation X / hybrid is deferred from the current main paper path
> - the trusted supervised anchor is the full-image `pretrained_resnet34_unet` baseline

## Cell 1 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected — go to Runtime > Change runtime type > GPU')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/foundation-nnunet'
assert os.path.exists(DRIVE_PROJECT), (
    f'Project not found at {DRIVE_PROJECT}. '
    'Upload your project folder to Google Drive first.'
)
print('Drive mounted. Project found at:', DRIVE_PROJECT)

## Cell 3 — Copy project to Colab local disk (fast I/O during training)

In [ ]:
import shutil, os

LOCAL = '/content/foundation-nnunet'

# Copy only src/, configs/, checkpoints/, results/, data/ on first run.
# On resume, preserve existing checkpoints/ so training continues.
if not os.path.exists(LOCAL):
    print('First run — copying project from Drive...')
    shutil.copytree(DRIVE_PROJECT, LOCAL)
    print('Done.')
else:
    print('Local copy exists — syncing src/ and configs/ only (preserving checkpoints)...')
    for folder in ['src', 'configs']:
        dst = os.path.join(LOCAL, folder)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(os.path.join(DRIVE_PROJECT, folder), dst)
    # Sync checkpoints FROM Drive (picks up any manually saved ones)
    drive_ckpts = os.path.join(DRIVE_PROJECT, 'checkpoints')
    local_ckpts = os.path.join(LOCAL, 'checkpoints')
    os.makedirs(local_ckpts, exist_ok=True)
    for f in os.listdir(drive_ckpts):
        src_f = os.path.join(drive_ckpts, f)
        dst_f = os.path.join(local_ckpts, f)
        if not os.path.exists(dst_f):   # don't overwrite newer local checkpoints
            shutil.copy2(src_f, dst_f)
            print(f'  Copied checkpoint from Drive: {f}')
    print('Done.')

os.chdir(LOCAL)
print('Working directory:', os.getcwd())

## Cell 4 — Install dependencies

In [ ]:
# Colab already has torch/torchvision. Install the rest.
!pip install -q timm albumentations pydicom scikit-learn tqdm pyyaml scipy opencv-python-headless
print('Dependencies installed.')

## Cell 5 — Verify data

In [ ]:
import json
from pathlib import Path

processed = Path('data/processed/pneumothorax')
splits_path = processed / 'splits.json'

assert splits_path.exists(), 'splits.json not found — run preprocess.py first'
with open(splits_path) as f:
    splits = json.load(f)

n_images = len(list((processed / 'images').glob('*.png')))
n_masks  = len(list((processed / 'masks').glob('*.png')))

print(f'Images: {n_images} | Masks: {n_masks}')
print(f'Train: {len(splits["train"])} | Val: {len(splits["val"])} | Test: {len(splits["test"])}')
assert n_images == n_masks, 'Image/mask count mismatch!'
print('Data OK.')

## Cell 6 — Configure: Baseline training

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['model']['type']            = 'baseline'
cfg['data']['input_size']       = 512
cfg['training']['batch_size']   = 8
cfg['training']['epochs']       = 150
cfg['device']                   = 'auto'
cfg['data']['num_workers']      = 2

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config set to BASELINE — 512x512, bs=8, 150 epochs')
print(yaml.dump(cfg, default_flow_style=False))

## Cell 7 — Train Baseline

In [ ]:
!python -m src.training.trainer --config configs/config.yaml

## Cell 8 — Save baseline results to Drive

In [ ]:
import shutil, os

drive_ckpts  = os.path.join(DRIVE_PROJECT, 'checkpoints')
drive_results = os.path.join(DRIVE_PROJECT, 'results')
os.makedirs(drive_ckpts,   exist_ok=True)
os.makedirs(drive_results, exist_ok=True)

for fname in ['best_baseline.pth', 'last_baseline.pth']:
    src = f'checkpoints/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_ckpts, fname))
        print(f'Saved {fname} → Drive')

for fname in ['baseline_history.csv']:
    src = f'results/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_results, fname))
        print(f'Saved {fname} → Drive')

print('Baseline results backed up to Drive.')

## Cell 9 — Configure: Hybrid training

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['model']['type']          = 'hybrid'
cfg['data']['input_size']     = 512
cfg['training']['batch_size'] = 4    # hybrid+512: reduced VRAM
cfg['training']['epochs']     = 150
cfg['device']                 = 'auto'
cfg['data']['num_workers']    = 2

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config set to HYBRID — 512x512, bs=4, 150 epochs')
print(yaml.dump(cfg, default_flow_style=False))

## Cell 10 — Train Hybrid

In [ ]:
!python -m src.training.trainer --config configs/config.yaml

## Cell 11 — Save hybrid results to Drive

In [ ]:
import shutil, os

drive_ckpts   = os.path.join(DRIVE_PROJECT, 'checkpoints')
drive_results = os.path.join(DRIVE_PROJECT, 'results')
os.makedirs(drive_ckpts,   exist_ok=True)
os.makedirs(drive_results, exist_ok=True)

for fname in ['best_hybrid.pth', 'last_hybrid.pth']:
    src = f'checkpoints/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_ckpts, fname))
        print(f'Saved {fname} → Drive')

for fname in ['hybrid_history.csv']:
    src = f'results/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_results, fname))
        print(f'Saved {fname} → Drive')

print('Hybrid results backed up to Drive.')

## Cell 12 — Evaluate both models on test set

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Restore eval-friendly settings
cfg['device'] = 'auto'
cfg['data']['num_workers'] = 2

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

!python -m src.evaluation.evaluate \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_baseline.pth \
    --model_type baseline

!python -m src.evaluation.evaluate \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_hybrid.pth \
    --model_type hybrid

## Cell 13 — Generate visualizations

In [ ]:
!python -m src.evaluation.visualize \
    --results_dir results/ \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_baseline.pth \
    --model_type baseline

# Show training curves inline
from IPython.display import Image, display
display(Image('results/training_curves.png'))
display(Image('results/model_comparison.png'))
display(Image('results/predictions_baseline.png'))

## Cell 14 — Save all final results to Drive

In [ ]:
import shutil, os
from pathlib import Path

drive_results = os.path.join(DRIVE_PROJECT, 'results')
os.makedirs(drive_results, exist_ok=True)

saved = []
for f in Path('results').glob('*'):
    shutil.copy2(str(f), os.path.join(drive_results, f.name))
    saved.append(f.name)

print('Saved to Drive/results/:')
for name in sorted(saved):
    print(f'  {name}')